In [ ]:
print("HellO")

In [ ]:
link="https://www.youtube.com/watch?v=Gjnup-PuquQ"



In [ ]:
import yt_dlp

ydl = yt_dlp.YoutubeDL({
    "quiet": True
})

info = ydl.extract_info(
    "ytsearch3:docker",
    download=False
)

for video in info["entries"]:
    print(video["title"])
    print(video["view_count"])
    print(video["duration"])

In [ ]:
type(info)

In [ ]:
for i in info['entries']:
    print(type(i))

In [ ]:
for video in info['entries']:
    print(video.keys())

In [ ]:
imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'average_rating', 'categories', 'tags', 'subtitles', 'like_count', 'webpage_url']

In [ ]:
from pprint import pprint

In [ ]:
for i in imp_params:
    print("\n\n\n", i, "\n")
    pprint(info['entries'][0][i])


In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(
    top_k_results=3,
    doc_content_chars_max=3000,
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

result = wiki_tool.invoke("Alan Turing")
print(result)

In [ ]:
import yt_dlp

class YTVideoFetcher:
    def __init__(self, topic, k=5):
        self.topic = topic
        self.k = k
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []



    def search_videos(self):

        self.results = ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):
        
    

In [8]:
from langchain_classic.prompts import PromptTemplate
from typing import Literal, List
from pydantic import Field, BaseModel
from langchain_groq import ChatGroq

import os

key = os.getenv("GROQ_API_KEY")
llm_llama = ChatGroq(model="llama-3.3-70b-versatile",api_key=key)

class Section(BaseModel):
    id: int = Field(description="Unique section number")
    title: str = Field(description="section title")
    importance: Literal["high", "medium", "low"]
    estimated_depth: Literal["brief", "medium", "detailed"]

class SectionPlan(BaseModel):
    sections: List[Section]


def subtopics_generator():
    user_query = 'I want to learn about reinforcement learning from zero to advanced'
    main_topic = 'Reinforcement learning'
    topic = 'Q-learning'
    structured_llm = llm_llama.with_structured_output(SectionPlan)

    

    prompt = PromptTemplate.from_template("""
    You are an expert technical content planner.

    Main Topic:
    {main_topic}

    Current Subtopic:
    {subtopic}

    User Query:
    {user_query}

    Your task is to decompose ONLY the current subtopic into a logical sequence of finer-grained sections that together form a complete explanation.

    Rules:
    - Generate between 5 and 12 sections.
    - Order them from introductory to advanced.
    - Avoid overlapping sections.
    - Focus only on the current subtopic.
    - Include mathematical concepts, algorithms, examples, and practical considerations when applicable.
    - Do not introduce unrelated concepts.

    Return ONLY valid JSON.

    {{
        "sections": [
            {{
                "id": 1,
                "title": "..."
            }}
        ]
    }}
    """)

    response = structured_llm.invoke(prompt.format(main_topic=main_topic, subtopic=topic, user_query=user_query))

    return response
    

In [9]:
resp = subtopics_generator()

In [23]:
resp.sections

[Section(id=1, title='Introduction to Q-learning', importance='high', estimated_depth='brief'),
 Section(id=2, title='Q-learning Algorithm', importance='high', estimated_depth='medium'),
 Section(id=3, title='Q-function and Action-value Function', importance='medium', estimated_depth='medium'),
 Section(id=4, title='Exploration-Exploitation Trade-off in Q-learning', importance='medium', estimated_depth='detailed'),
 Section(id=5, title='Q-learning Convergence and Optimality', importance='medium', estimated_depth='detailed'),
 Section(id=6, title='Deep Q-Networks (DQN) and Q-learning', importance='low', estimated_depth='detailed'),
 Section(id=7, title='Q-learning in Continuous Action Spaces', importance='low', estimated_depth='detailed'),
 Section(id=8, title='Practical Considerations for Implementing Q-learning', importance='high', estimated_depth='medium')]